In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 — Install / upgrade all required packages
#           Re-run this cell if you restart the Colab runtime.
# ─────────────────────────────────────────────────────────────────────────────
!pip install -q flask flask-cors deepface mtcnn tf-keras pillow numpy
# !pip install -q cloudflared          # Python wrapper that ships the binary

In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║              FACE BROWSER — GOOGLE COLAB SERVER (colab_face_server.py)       ║
║                                                                              ║
║  Endpoints:                                                                  ║
║    GET  /api/health         — liveness probe + DB size                       ║
║    POST /api/upload         — detect faces, return face 0 frame+thumb        ║
║    POST /api/render_all     — kick off background render of all frames       ║
║    GET  /api/render_status  — poll for newly-ready frames + thumbs           ║
║    POST /api/set_display_px — change resolution, re-render everything        ║
║    POST /api/select         — return frame+thumb for a specific face         ║
║    POST /api/add_to_db      — embed + store a face in FACE_DB                ║
║    POST /api/identify       — cosine-search FACE_DB for best match           ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""

# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 — Install dependencies (re-run after every Colab runtime restart)
# ─────────────────────────────────────────────────────────────────────────────
# !pip install -q flask flask-cors deepface mtcnn tf-keras pillow numpy


# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 — Imports & configuration
# ─────────────────────────────────────────────────────────────────────────────

import io
import os
import uuid
import base64
import threading
import subprocess
import time
import re
import socket

import numpy as np
from PIL import Image, ImageOps, ImageDraw
from flask import Flask, request, jsonify
from flask_cors import CORS
from deepface import DeepFace as df

# ── Server-side defaults (overridable per-request via slider form fields) ────
DETECT_MAX_PX  = 1280   # long-edge px cap used during MTCNN detection; -1 = no cap
DISPLAY_MAX_PX = 900    # long-edge px cap for annotated preview PNGs
THUMB_SIZE     = 200    # square px for face thumbnail crops
FLASK_PORT     = 5000

# ── Session store ─────────────────────────────────────────────────────────────
# Maps session_id (str UUID) → session dict.
# Full schema documented on the upload route.
_SESSIONS: dict = {}


# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 — Image-processing helpers
# ─────────────────────────────────────────────────────────────────────────────

def resize_keep_aspect(img: Image.Image, max_px: int):
    """
    Downscale *img* so its longest edge ≤ *max_px*, preserving aspect ratio.
    Returns (resized_image, scale_factor).
    scale_factor == 1.0 means the image was already small enough.
    max_px == -1 → return image unchanged with scale 1.0.
    """
    if max_px == -1:
        return img, 1.0
    w, h  = img.size
    scale = min(max_px / w, max_px / h, 1.0)
    if scale >= 1.0:
        return img, 1.0
    return img.resize((int(w * scale), int(h * scale)), Image.LANCZOS), scale


def pil_to_png_b64(img: Image.Image) -> str:
    """Encode a PIL image as a raw base64 PNG string (no data-URI prefix)."""
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode()


def arr_to_pil(arr: np.ndarray) -> Image.Image:
    """Convert a DeepFace float32 face array (H×W×3, values 0–1) to uint8 PIL."""
    if arr.dtype != np.uint8:
        arr = (np.clip(arr, 0, 1) * 255).astype(np.uint8)
    return Image.fromarray(arr)


def _ts() -> str:
    """Return a short timestamp string for debug prints, e.g. '12:34:56.789'."""
    return time.strftime("%H:%M:%S") + f".{int(time.time() * 1000) % 1000:03d}"


def _sid(session_id: str) -> str:
    """Return a shortened session ID for legible debug output."""
    return session_id[:8] if session_id else "????????"


# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 — Face detection
# ─────────────────────────────────────────────────────────────────────────────

def get_face_data(image: Image.Image, detect_max_px: int = DETECT_MAX_PX):
    """
    Run MTCNN face detection on *image*.

    If detect_max_px < the image's longest edge, detection runs on a downscaled
    copy for speed, then bounding-box coordinates and face crops are mapped back
    to the original full-resolution image so thumbnails stay sharp.

    Returns (exif_corrected_image, faceobjs_in_original_px_space).
    """
    image = ImageOps.exif_transpose(image)
    detect_img, scale = resize_keep_aspect(image, detect_max_px)
    detect_path = "/tmp/_detect_scaled.jpg"
    detect_img.save(detect_path, quality=90)

    print(f"[{_ts()}] get_face_data: running MTCNN on "
          f"{detect_img.size[0]}×{detect_img.size[1]}px (scale={scale:.3f}, "
          f"detect_max_px={detect_max_px})")

    faceobjs = df.extract_faces(detect_path, detector_backend="mtcnn")

    print(f"[{_ts()}] get_face_data: found {len(faceobjs)} face(s)")

    # Map bounding boxes back to original full-res space if we downscaled
    if scale < 1.0:
        for face in faceobjs:
            r = face["facial_area"]
            r["x"] = int(r["x"] / scale); r["y"] = int(r["y"] / scale)
            r["w"] = int(r["w"] / scale); r["h"] = int(r["h"] / scale)
            x, y, w, h = r["x"], r["y"], r["w"], r["h"]
            crop = image.crop((x, y, x + w, y + h))
            face["face"] = np.array(crop).astype(np.float32) / 255.0

    return image, faceobjs


# ─────────────────────────────────────────────────────────────────────────────
# CELL 5 — Rendering helpers
# ─────────────────────────────────────────────────────────────────────────────

def render_frame(display_img: Image.Image, display_faceobjs: list, sel: int) -> str:
    """
    Draw bounding boxes on a copy of *display_img* and return it as base64 PNG.
    Selected face (index *sel*) gets a 5-rect cyan glow; others get a red outline.
    """
    img = display_img.copy()
    d   = ImageDraw.Draw(img)
    for i, face in enumerate(display_faceobjs):
        r = face["facial_area"]
        x, y, w, h = r["x"], r["y"], r["w"], r["h"]
        if i == sel:
            for off in range(5):
                d.rectangle([x - off, y - off, x + w + off, y + h + off], outline="#00ffcc")
        else:
            d.rectangle([x, y, x + w, y + h], outline="#ff4444", width=2)
    return pil_to_png_b64(img)


def make_thumb(face_arr: np.ndarray, size: int) -> str:
    """Convert a DeepFace float32 face array to a square thumbnail PNG (base64)."""
    pil = arr_to_pil(face_arr)
    if size != -1:
        pil = pil.resize((size, size), Image.LANCZOS)
    pil = ImageOps.expand(pil, border=3, fill="#00ffcc")
    return pil_to_png_b64(pil)


def build_display_faceobjs(faceobjs: list, disp_scale: float) -> list:
    """
    Return a copy of *faceobjs* with facial_area coords scaled to display-image space.
    Used for overlay rendering only — identification uses full-res face arrays.
    """
    return [{
        **face,
        "facial_area": {
            "x": int(face["facial_area"]["x"] * disp_scale),
            "y": int(face["facial_area"]["y"] * disp_scale),
            "w": int(face["facial_area"]["w"] * disp_scale),
            "h": int(face["facial_area"]["h"] * disp_scale),
        }
    } for face in faceobjs]


def face_areas_from_display_faceobjs(display_faceobjs: list) -> list:
    """Extract a plain [{x,y,w,h}] list from display_faceobjs for the client."""
    return [{k: f["facial_area"][k] for k in ("x","y","w","h")} for f in display_faceobjs]


# ─────────────────────────────────────────────────────────────────────────────
# CELL 6 — In-memory face database
# ─────────────────────────────────────────────────────────────────────────────

FACE_DB: list[dict] = []   # [{"label": str, "embedding": list[float]}, …]
DB_LOCK = threading.Lock()


def cosine_similarity(a, b) -> float:
    a, b  = np.array(a), np.array(b)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom > 0 else 0.0


# ─────────────────────────────────────────────────────────────────────────────
# CELL 7 — Background render worker
# ─────────────────────────────────────────────────────────────────────────────

def _render_worker(session_id: str):
    """
    Background daemon thread: render every face-selection frame AND thumbnail
    for the session and store results in render_cache / thumb_cache.

    Design notes
    ────────────
    • Skips face 0 — it was pre-rendered synchronously during /api/upload or
      /api/set_display_px and is already in the cache.
    • Uses render_generation to detect if a /api/set_display_px call has
      invalidated this worker mid-run.  If the generation has changed, the
      worker exits without writing stale data.
    • Sets session["rendering_idx"] to the currently-in-progress index so
      /api/render_status can report it to the client as "rendering_now".
    • Clears rendering_idx to None between faces and on exit.
    """
    session = _SESSIONS.get(session_id)
    if not session:
        print(f"[{_ts()}] _render_worker [{_sid(session_id)}]: session not found, exiting")
        return

    my_gen   = session.get("render_generation", 0)
    n_faces  = len(session["faceobjs"])

    print(f"[{_ts()}] _render_worker [{_sid(session_id)}]: starting, "
          f"{n_faces} faces, generation={my_gen}")

    for i in range(n_faces):
        # ── Superseded check — abort if a newer set_display_px fired ─────────
        if session.get("render_generation", 0) != my_gen:
            print(f"[{_ts()}] _render_worker [{_sid(session_id)}]: "
                  f"generation changed at face {i}, exiting")
            return

        # ── Skip already-cached faces ─────────────────────────────────────────
        with session["render_lock"]:
            if session["render_cache"][i] is not None:
                continue
            session["rendering_idx"] = i

        t0 = time.time()
        print(f"[{_ts()}] _render_worker [{_sid(session_id)}]: rendering face {i} …")

        # Render the annotated frame (CPU-intensive — done outside the lock)
        frame_b64 = render_frame(
            session["display_img"],
            session["display_faceobjs"],
            i,
        )
        # Render the thumbnail (fast)
        thumb_b64 = make_thumb(session["faceobjs"][i]["face"], THUMB_SIZE)

        elapsed = time.time() - t0

        # ── Final generation check before writing ─────────────────────────────
        with session["render_lock"]:
            if session.get("render_generation", 0) != my_gen:
                print(f"[{_ts()}] _render_worker [{_sid(session_id)}]: "
                      f"generation changed while writing face {i}, exiting")
                session["rendering_idx"] = None
                return
            session["render_cache"][i] = frame_b64
            session["thumb_cache"][i]  = thumb_b64
            session["rendering_idx"]   = None

        print(f"[{_ts()}] _render_worker [{_sid(session_id)}]: "
              f"face {i} done in {elapsed:.2f}s")

    print(f"[{_ts()}] _render_worker [{_sid(session_id)}]: "
          f"all {n_faces} faces complete")


def _reset_render_cache(session: dict):
    """
    Clear render_cache, thumb_cache, and delivered_indices, then bump
    render_generation to invalidate any currently-running _render_worker.
    Called by /api/set_display_px before starting a fresh worker.
    """
    with session["render_lock"]:
        n = len(session["faceobjs"])
        session["render_cache"]      = [None] * n
        session["thumb_cache"]       = [None] * n
        session["delivered_indices"] = set()
        session["rendering_idx"]     = None
        session["render_generation"] = session.get("render_generation", 0) + 1
    print(f"[{_ts()}] _reset_render_cache: cache cleared, "
          f"new generation={session['render_generation']}")


# ─────────────────────────────────────────────────────────────────────────────
# CELL 8 — Flask application
# ─────────────────────────────────────────────────────────────────────────────

app = Flask(__name__)
CORS(app, resources={r"/api/*": {"origins": "*"}})


# ── GET /api/health ────────────────────────────────────────────────────────────

@app.route("/api/health", methods=["GET"])
def health():
    """Liveness probe. Returns current FACE_DB entry count."""
    print(f"[{_ts()}] GET /api/health  →  db_size={len(FACE_DB)}")
    return jsonify({"status": "ok", "face_db_size": len(FACE_DB)})


# ── POST /api/upload ───────────────────────────────────────────────────────────

@app.route("/api/upload", methods=["POST"])
def upload():
    """
    Accept a multipart image upload, run face detection, and return face 0's
    annotated frame + thumbnail immediately.

    Session schema created here:
      image, faceobjs, display_img, display_faceobjs, display_max_px,
      render_cache[i]  — base64 PNG or None (face 0 pre-filled)
      thumb_cache[i]   — base64 PNG or None (face 0 pre-filled)
      render_lock, rendering_idx, delivered_indices, render_generation

    Form fields:
      image          — image file (JPEG/PNG/WEBP …)
      detect_max_px  — int string (optional)
      display_max_px — int string (optional)
    """
    print(f"[{_ts()}] POST /api/upload  content_length={request.content_length}")

    if "image" not in request.files:
        print(f"[{_ts()}] POST /api/upload  ERROR: no image field")
        return jsonify({"error": "No image field in request"}), 400

    detect_max_px  = int(request.form.get("detect_max_px",  DETECT_MAX_PX))
    display_max_px = int(request.form.get("display_max_px", DISPLAY_MAX_PX))
    print(f"[{_ts()}] POST /api/upload  detect_max_px={detect_max_px}  "
          f"display_max_px={display_max_px}")

    file = request.files["image"]
    try:
        image = Image.open(file.stream).convert("RGB")
        print(f"[{_ts()}] POST /api/upload  image opened: {image.size[0]}×{image.size[1]}px")
    except Exception as exc:
        print(f"[{_ts()}] POST /api/upload  ERROR opening image: {exc}")
        return jsonify({"error": f"Cannot open image: {exc}"}), 400

    try:
        original_image, faceobjs = get_face_data(image, detect_max_px)
    except Exception as exc:
        print(f"[{_ts()}] POST /api/upload  ERROR in detection: {exc}")
        return jsonify({"error": f"Detection failed: {exc}"}), 500

    if not faceobjs:
        print(f"[{_ts()}] POST /api/upload  no faces detected")
        return jsonify({"error": "No faces detected in this image"}), 200

    display_img, disp_scale = resize_keep_aspect(original_image, display_max_px)
    display_faceobjs        = build_display_faceobjs(faceobjs, disp_scale)
    disp_w, disp_h          = display_img.size
    print(f"[{_ts()}] POST /api/upload  display_img={disp_w}×{disp_h}px  "
          f"disp_scale={disp_scale:.3f}  faces={len(faceobjs)}")

    # Pre-render face 0 synchronously so the UI can show something immediately
    t0 = time.time()
    frame0_b64 = render_frame(display_img, display_faceobjs, 0)
    thumb0_b64 = make_thumb(faceobjs[0]["face"], THUMB_SIZE)
    print(f"[{_ts()}] POST /api/upload  face 0 rendered in {time.time()-t0:.2f}s")

    # Initialise caches: face 0 pre-filled, rest None
    n = len(faceobjs)
    render_cache       = [None] * n;  render_cache[0] = frame0_b64
    thumb_cache        = [None] * n;  thumb_cache[0]  = thumb0_b64

    session_id = str(uuid.uuid4())
    _SESSIONS[session_id] = {
        "image":             original_image,
        "faceobjs":          faceobjs,
        "display_img":       display_img,
        "display_faceobjs":  display_faceobjs,
        "display_max_px":    display_max_px,
        # Render caches — both indexed by face index
        "render_cache":      render_cache,   # frame PNGs
        "thumb_cache":       thumb_cache,    # thumbnail PNGs
        "render_lock":       threading.Lock(),
        "rendering_idx":     None,
        "delivered_indices": set(),          # frames already sent to the client
        "render_generation": 0,
    }

    conf = faceobjs[0].get("confidence")
    print(f"[{_ts()}] POST /api/upload  session={_sid(session_id)}  "
          f"conf={conf:.4f if conf else 'N/A'}")

    return jsonify({
        "session_id": session_id,
        "face_count": n,
        "frame_b64":  frame0_b64,
        "thumb_b64":  thumb0_b64,
        "confidence": round(conf, 4) if conf else None,
        "disp_w":     disp_w,
        "disp_h":     disp_h,
        "face_areas": face_areas_from_display_faceobjs(display_faceobjs),
    })


# ── POST /api/render_all ───────────────────────────────────────────────────────

@app.route("/api/render_all", methods=["POST"])
def render_all():
    """
    Start background rendering of ALL face frames + thumbnails for the session.
    Returns immediately — rendering is non-blocking.

    Request JSON:  { "session_id": "<uuid>" }
    Response JSON: { "status": "started", "face_count": N }
    """
    data       = request.get_json(force=True)
    session_id = data.get("session_id", "")
    session    = _SESSIONS.get(session_id)

    print(f"[{_ts()}] POST /api/render_all  session={_sid(session_id)}")

    if not session:
        print(f"[{_ts()}] POST /api/render_all  ERROR: unknown session")
        return jsonify({"error": "Unknown session_id"}), 404

    n = len(session["faceobjs"])
    print(f"[{_ts()}] POST /api/render_all  launching worker for {n} faces")

    t = threading.Thread(target=_render_worker, args=(session_id,), daemon=True)
    t.start()

    return jsonify({"status": "started", "face_count": n})


# ── GET /api/render_status ────────────────────────────────────────────────────

@app.route("/api/render_status", methods=["GET"])
def render_status():
    """
    Return which frames finished rendering since the last poll, along with
    their frame AND thumbnail base64 data (delivered only once per face).

    The server tracks session["delivered_indices"] so each face's image data
    is sent exactly once — subsequent polls return only the index metadata,
    keeping responses tiny.

    Query params:  ?session_id=<uuid>

    Response JSON:
    {
      "ready":         [0, 2, …],
      "rendering_now": [1],          // currently in-progress index, or []
      "total":         N,
      "frames": {                    // NEW since last poll only
        "2": { "frame_b64": "…", "thumb_b64": "…" }
      }
    }
    """
    session_id = request.args.get("session_id", "")
    session    = _SESSIONS.get(session_id)

    if not session:
        return jsonify({"error": "Unknown session_id"}), 404

    render_lock   = session["render_lock"]
    render_cache  = session["render_cache"]
    thumb_cache   = session["thumb_cache"]
    delivered     = session["delivered_indices"]
    rendering_idx = session.get("rendering_idx")

    new_frames = {}
    ready_list = []

    with render_lock:
        for i, frame_b64 in enumerate(render_cache):
            if frame_b64 is not None:
                ready_list.append(i)
                if i not in delivered:
                    new_frames[str(i)] = {
                        "frame_b64": frame_b64,
                        "thumb_b64": thumb_cache[i],   # always set when frame is set
                    }
                    delivered.add(i)

    if new_frames:
        print(f"[{_ts()}] GET /api/render_status  session={_sid(session_id)}  "
              f"ready={ready_list}  new_frames={list(new_frames.keys())}  "
              f"rendering_now={rendering_idx}")
    # (suppress noisy no-op polls to keep log clean)

    return jsonify({
        "ready":         ready_list,
        "rendering_now": [rendering_idx] if rendering_idx is not None else [],
        "total":         len(render_cache),
        "frames":        new_frames,
    })


# ── POST /api/set_display_px ───────────────────────────────────────────────────

@app.route("/api/set_display_px", methods=["POST"])
def set_display_px():
    """
    Change the display resolution for an existing session and re-render everything.

    Steps:
    1. Rebuild display_img + display_faceobjs at the new resolution.
    2. Invalidate the render cache (_reset_render_cache bumps generation).
    3. Immediately render the currently-selected face at the new resolution.
    4. Seed that frame + thumb into the cache and mark as delivered.
    5. Start a new _render_worker for all remaining faces.

    Request JSON:
      { "session_id", "display_max_px", "current_face_index" }

    Response JSON:
      { "disp_w", "disp_h", "current_frame_b64", "thumb_b64",
        "confidence", "face_areas" }
    """
    data               = request.get_json(force=True)
    session_id         = data.get("session_id", "")
    new_display_max_px = int(data.get("display_max_px", DISPLAY_MAX_PX))
    current_idx        = int(data.get("current_face_index", 0))

    print(f"[{_ts()}] POST /api/set_display_px  session={_sid(session_id)}  "
          f"display_max_px={new_display_max_px}  current_face={current_idx}")

    session = _SESSIONS.get(session_id)
    if not session:
        print(f"[{_ts()}] POST /api/set_display_px  ERROR: unknown session")
        return jsonify({"error": "Unknown session_id"}), 404

    faceobjs    = session["faceobjs"]
    current_idx = max(0, min(current_idx, len(faceobjs) - 1))

    # Rebuild display image at the new resolution
    original_image          = session["image"]
    display_img, disp_scale = resize_keep_aspect(original_image, new_display_max_px)
    display_faceobjs        = build_display_faceobjs(faceobjs, disp_scale)
    disp_w, disp_h          = display_img.size
    print(f"[{_ts()}] POST /api/set_display_px  new display={disp_w}×{disp_h}px  "
          f"disp_scale={disp_scale:.3f}")

    session["display_img"]      = display_img
    session["display_faceobjs"] = display_faceobjs
    session["display_max_px"]   = new_display_max_px

    # Flush stale cache + bump generation so old worker exits
    _reset_render_cache(session)

    # Immediately render the current face so the UI can show something right away
    t0 = time.time()
    frame_b64 = render_frame(display_img, display_faceobjs, current_idx)
    thumb_b64 = make_thumb(faceobjs[current_idx]["face"], THUMB_SIZE)
    print(f"[{_ts()}] POST /api/set_display_px  face {current_idx} rendered "
          f"in {time.time()-t0:.2f}s")

    with session["render_lock"]:
        session["render_cache"][current_idx] = frame_b64
        session["thumb_cache"][current_idx]  = thumb_b64
        # Mark as delivered so the poll loop won't re-send it
        session["delivered_indices"].add(current_idx)

    # Start background rendering for all remaining faces
    t = threading.Thread(target=_render_worker, args=(session_id,), daemon=True)
    t.start()
    print(f"[{_ts()}] POST /api/set_display_px  background worker started")

    conf = faceobjs[current_idx].get("confidence")
    return jsonify({
        "disp_w":            disp_w,
        "disp_h":            disp_h,
        "current_frame_b64": frame_b64,
        "thumb_b64":         thumb_b64,
        "confidence":        round(conf, 4) if conf else None,
        "face_areas":        face_areas_from_display_faceobjs(display_faceobjs),
    })


# ── POST /api/select ───────────────────────────────────────────────────────────

@app.route("/api/select", methods=["POST"])
def select_face():
    """
    Return the frame + thumbnail for the selected face index.

    Serves from render_cache / thumb_cache if available (avoids re-rendering).
    Falls back to a fresh render for uncached faces (shouldn't happen normally
    since the client guards on readySet, but handles edge cases gracefully).

    Request JSON:  { "session_id": "<uuid>", "face_index": <int> }
    Response JSON: { "frame_b64", "thumb_b64", "confidence" }
    """
    data       = request.get_json(force=True)
    session_id = data.get("session_id", "")
    idx        = int(data.get("face_index", 0))
    session    = _SESSIONS.get(session_id)

    print(f"[{_ts()}] POST /api/select  session={_sid(session_id)}  face={idx}")

    if not session:
        print(f"[{_ts()}] POST /api/select  ERROR: unknown session")
        return jsonify({"error": "Unknown session_id"}), 404

    faceobjs         = session["faceobjs"]
    display_img      = session["display_img"]
    display_faceobjs = session["display_faceobjs"]
    idx = max(0, min(idx, len(faceobjs) - 1))

    with session["render_lock"]:
        cached_frame = session["render_cache"][idx]
        cached_thumb = session["thumb_cache"][idx]

    if cached_frame and cached_thumb:
        print(f"[{_ts()}] POST /api/select  serving face {idx} from cache")
        frame_b64 = cached_frame
        thumb_b64 = cached_thumb
    else:
        print(f"[{_ts()}] POST /api/select  cache miss for face {idx}, rendering now")
        t0        = time.time()
        frame_b64 = render_frame(display_img, display_faceobjs, idx)
        thumb_b64 = make_thumb(faceobjs[idx]["face"], THUMB_SIZE)
        print(f"[{_ts()}] POST /api/select  face {idx} rendered in {time.time()-t0:.2f}s")

    conf = faceobjs[idx].get("confidence")
    return jsonify({
        "frame_b64": frame_b64,
        "thumb_b64": thumb_b64,
        "confidence": round(conf, 4) if conf else None,
    })


# ── POST /api/add_to_db ────────────────────────────────────────────────────────

@app.route("/api/add_to_db", methods=["POST"])
def add_to_db():
    """
    Embed the selected face with Facenet512 and store it in FACE_DB.

    Request JSON:  { "session_id", "face_index", "label" }
    Response JSON: { "message", "db_size" }
    """
    data       = request.get_json(force=True)
    session_id = data.get("session_id", "")
    idx        = int(data.get("face_index", 0))
    label      = data.get("label", f"Person_{len(FACE_DB)+1}")

    print(f"[{_ts()}] POST /api/add_to_db  session={_sid(session_id)}  "
          f"face={idx}  label='{label}'")

    session = _SESSIONS.get(session_id)
    if not session:
        print(f"[{_ts()}] POST /api/add_to_db  ERROR: unknown session")
        return jsonify({"error": "Unknown session_id"}), 404

    faceobjs = session["faceobjs"]
    idx      = max(0, min(idx, len(faceobjs) - 1))
    face_arr = faceobjs[idx]["face"]

    tmp_path = f"/tmp/_face_{uuid.uuid4().hex}.jpg"
    arr_to_pil(face_arr).save(tmp_path)

    t0 = time.time()
    try:
        result    = df.represent(tmp_path, model_name="Facenet512", detector_backend="skip")
        embedding = result[0]["embedding"]
        print(f"[{_ts()}] POST /api/add_to_db  embedding done in {time.time()-t0:.2f}s  "
              f"dim={len(embedding)}")
    except Exception as exc:
        print(f"[{_ts()}] POST /api/add_to_db  ERROR embedding: {exc}")
        return jsonify({"error": f"Embedding failed: {exc}"}), 500
    finally:
        os.remove(tmp_path)

    with DB_LOCK:
        FACE_DB.append({"label": label, "embedding": embedding})
        db_size = len(FACE_DB)

    print(f"[{_ts()}] POST /api/add_to_db  '{label}' added — db_size={db_size}")
    return jsonify({"message": f"✅ '{label}' added to database.", "db_size": db_size})


# ── POST /api/identify ─────────────────────────────────────────────────────────

@app.route("/api/identify", methods=["POST"])
def identify():
    """
    Embed the selected face and cosine-search FACE_DB for the best match.
    Reports "Unknown" if best similarity < 0.70.

    Request JSON:  { "session_id", "face_index" }
    Response JSON: { "match", "similarity", "message" }
    """
    data       = request.get_json(force=True)
    session_id = data.get("session_id", "")
    idx        = int(data.get("face_index", 0))

    print(f"[{_ts()}] POST /api/identify  session={_sid(session_id)}  face={idx}")

    session = _SESSIONS.get(session_id)
    if not session:
        print(f"[{_ts()}] POST /api/identify  ERROR: unknown session")
        return jsonify({"error": "Unknown session_id"}), 404

    with DB_LOCK:
        if not FACE_DB:
            print(f"[{_ts()}] POST /api/identify  database is empty")
            return jsonify({"match": "Unknown", "similarity": 0.0,
                            "message": "⚠️ Database is empty — add some faces first."})

    faceobjs = session["faceobjs"]
    idx      = max(0, min(idx, len(faceobjs) - 1))
    face_arr = faceobjs[idx]["face"]

    tmp_path = f"/tmp/_face_{uuid.uuid4().hex}.jpg"
    arr_to_pil(face_arr).save(tmp_path)

    t0 = time.time()
    try:
        result    = df.represent(tmp_path, model_name="Facenet512", detector_backend="skip")
        query_emb = result[0]["embedding"]
        print(f"[{_ts()}] POST /api/identify  embedding done in {time.time()-t0:.2f}s")
    except Exception as exc:
        print(f"[{_ts()}] POST /api/identify  ERROR embedding: {exc}")
        return jsonify({"error": f"Embedding failed: {exc}"}), 500
    finally:
        os.remove(tmp_path)

    with DB_LOCK:
        best_label, best_sim = "Unknown", -1.0
        for entry in FACE_DB:
            sim = cosine_similarity(query_emb, entry["embedding"])
            if sim > best_sim:
                best_sim, best_label = sim, entry["label"]

    THRESHOLD = 0.70
    if best_sim < THRESHOLD:
        best_label = "Unknown"

    print(f"[{_ts()}] POST /api/identify  best_match='{best_label}'  "
          f"similarity={best_sim:.4f}  threshold={THRESHOLD}")

    return jsonify({
        "match":      best_label,
        "similarity": round(best_sim, 4),
        "message":    f"🔍 Best match: {best_label} (similarity {best_sim:.3f})",
    })


# ─────────────────────────────────────────────────────────────────────────────
# CELL 9 — Cloudflare Quick Tunnel
# ─────────────────────────────────────────────────────────────────────────────

CLOUDFLARED_BIN = "/usr/local/bin/cloudflared"
CLOUDFLARED_URL = (
    "https://github.com/cloudflare/cloudflared/releases/latest/"
    "download/cloudflared-linux-amd64"
)


def ensure_cloudflared() -> str:
    if os.path.isfile(CLOUDFLARED_BIN) and os.access(CLOUDFLARED_BIN, os.X_OK):
        print(f"[{_ts()}] cloudflared already installed at {CLOUDFLARED_BIN}")
        return CLOUDFLARED_BIN
    print(f"[{_ts()}] Downloading cloudflared binary…")
    result = subprocess.run(["wget", "-q", "-O", CLOUDFLARED_BIN, CLOUDFLARED_URL],
                            capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"wget failed: {result.stderr}")
    os.chmod(CLOUDFLARED_BIN, 0o755)
    print(f"[{_ts()}] cloudflared installed → {CLOUDFLARED_BIN}")
    return CLOUDFLARED_BIN


def start_cloudflare_tunnel(port: int, timeout: int = 60) -> str | None:
    """Launch cloudflared and parse the *.trycloudflare.com URL from stderr."""
    bin_path    = ensure_cloudflared()
    proc        = subprocess.Popen(
        [bin_path, "tunnel", "--url", f"http://localhost:{port}"],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True,
    )
    url_pattern = re.compile(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com")
    deadline    = time.time() + timeout
    while time.time() < deadline:
        line = proc.stderr.readline()
        if not line:
            print(f"[{_ts()}] ⚠️  cloudflared process exited unexpectedly.")
            break
        match = url_pattern.search(line)
        if match:
            return match.group(0)
    print(f"[{_ts()}] ⚠️  Timed out waiting for Cloudflare tunnel URL.")
    return None


# ─────────────────────────────────────────────────────────────────────────────
# CELL 10 — Entry point
# ─────────────────────────────────────────────────────────────────────────────
def find_free_port():
  global FLASK_PORT
  while True:
      with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
          # connect_ex returns 0 if the port is open/busy
          if s.connect_ex(('0.0.0.0', FLASK_PORT)) != 0:
              return FLASK_PORT
          FLASK_PORT += 1

def run_server():
    """Start Flask dev server in a daemon thread."""
    app.run(host="0.0.0.0", port=find_free_port(), use_reloader=False)


if __name__ == "__main__" or True:
    server_thread = threading.Thread(target=run_server, daemon=True)
    server_thread.start()
    print(f"[{_ts()}] Flask server started on http://localhost:{FLASK_PORT}")
    time.sleep(1)

    print(f"[{_ts()}] Opening Cloudflare Quick Tunnel… (may take ~15 s)")
    public_url = start_cloudflare_tunnel(FLASK_PORT)

    if public_url:
        print("\n" + "═" * 60)
        print(f"  ✅  API is live at:\n      {public_url}")
        print(f"\n  Paste this URL into the GitHub Pages UI.")
        print("═" * 60 + "\n")
    else:
        print("❌  Could not obtain tunnel URL.")

In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║              FACE BROWSER — GOOGLE COLAB SERVER (colab_face_server.py)      ║
║                                                                              ║
║  Endpoints:                                                                  ║
║    GET  /api/health         — liveness probe + DB size                      ║
║    POST /api/upload         — detect faces, return face 0 frame+thumb       ║
║    POST /api/render_all     — kick off background render of all frames      ║
║    GET  /api/render_status  — poll for newly-ready frames + thumbs          ║
║    POST /api/set_display_px — change resolution, re-render everything       ║
║    POST /api/select         — return frame+thumb for a specific face        ║
║    POST /api/add_to_db      — embed + store a face in FACE_DB               ║
║    POST /api/identify       — cosine-search FACE_DB for best match          ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""

# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 — Install dependencies (re-run after every Colab runtime restart)
# ─────────────────────────────────────────────────────────────────────────────
# !pip install -q flask flask-cors deepface mtcnn tf-keras pillow numpy


# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 — Imports & configuration
# ─────────────────────────────────────────────────────────────────────────────

import io
import os
import uuid
import base64
import threading
import subprocess
import time
import re
import socket

import numpy as np
from PIL import Image, ImageOps, ImageDraw
from flask import Flask, request, jsonify
from flask_cors import CORS
from deepface import DeepFace as df

# ── Server-side defaults (overridable per-request via slider form fields) ────
DETECT_MAX_PX  = 1280   # long-edge px cap used during MTCNN detection; -1 = no cap
DISPLAY_MAX_PX = 900    # long-edge px cap for annotated preview PNGs
THUMB_SIZE     = 200    # square px for face thumbnail crops
FLASK_PORT     = 5000

# ── Session store ─────────────────────────────────────────────────────────────
# Maps session_id (str UUID) → session dict.
# Full schema documented on the upload route.
_SESSIONS: dict = {}
# ── Globals ───────────────────────────────────────────────────────────────────
STOP_EVENT = threading.Event()
CLOUDFLARE_PROC = None

# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 — Image-processing helpers
# ─────────────────────────────────────────────────────────────────────────────

def resize_keep_aspect(img: Image.Image, max_px: int):
    """
    Downscale *img* so its longest edge ≤ *max_px*, preserving aspect ratio.
    Returns (resized_image, scale_factor).
    scale_factor == 1.0 means the image was already small enough.
    max_px == -1 → return image unchanged with scale 1.0.
    """
    if max_px == -1:
        return img, 1.0
    w, h  = img.size
    scale = min(max_px / w, max_px / h, 1.0)
    if scale >= 1.0:
        return img, 1.0
    return img.resize((int(w * scale), int(h * scale)), Image.LANCZOS), scale


def pil_to_png_b64(img: Image.Image) -> str:
    """Encode a PIL image as a raw base64 PNG string (no data-URI prefix)."""
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode()


def arr_to_pil(arr: np.ndarray) -> Image.Image:
    """Convert a DeepFace float32 face array (H×W×3, values 0–1) to uint8 PIL."""
    if arr.dtype != np.uint8:
        arr = (np.clip(arr, 0, 1) * 255).astype(np.uint8)
    return Image.fromarray(arr)


def _ts() -> str:
    """Return a short timestamp string for debug prints, e.g. '12:34:56.789'."""
    return time.strftime("%H:%M:%S") + f".{int(time.time() * 1000) % 1000:03d}"


def _sid(session_id: str) -> str:
    """Return a shortened session ID for legible debug output."""
    return session_id[:8] if session_id else "????????"


# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 — Face detection
# ─────────────────────────────────────────────────────────────────────────────

def get_face_data(image: Image.Image, detect_max_px: int = DETECT_MAX_PX):
    """
    Run MTCNN face detection on *image*.

    If detect_max_px < the image's longest edge, detection runs on a downscaled
    copy for speed, then bounding-box coordinates and face crops are mapped back
    to the original full-resolution image so thumbnails stay sharp.

    Returns (exif_corrected_image, faceobjs_in_original_px_space).
    """
    image = ImageOps.exif_transpose(image)
    detect_img, scale = resize_keep_aspect(image, detect_max_px)
    detect_path = "/tmp/_detect_scaled.jpg"
    detect_img.save(detect_path, quality=90)

    print(f"[{_ts()}] get_face_data: running MTCNN on "
          f"{detect_img.size[0]}×{detect_img.size[1]}px (scale={scale:.3f}, "
          f"detect_max_px={detect_max_px})")

    faceobjs = df.extract_faces(detect_path, detector_backend="mtcnn")

    print(f"[{_ts()}] get_face_data: found {len(faceobjs)} face(s)")

    # Map bounding boxes back to original full-res space if we downscaled
    if scale < 1.0:
        for face in faceobjs:
            r = face["facial_area"]
            r["x"] = int(r["x"] / scale); r["y"] = int(r["y"] / scale)
            r["w"] = int(r["w"] / scale); r["h"] = int(r["h"] / scale)
            x, y, w, h = r["x"], r["y"], r["w"], r["h"]
            crop = image.crop((x, y, x + w, y + h))
            face["face"] = np.array(crop).astype(np.float32) / 255.0

    return image, faceobjs


# ─────────────────────────────────────────────────────────────────────────────
# CELL 5 — Rendering helpers
# ─────────────────────────────────────────────────────────────────────────────

def render_frame(display_img: Image.Image, display_faceobjs: list, sel: int) -> str:
    """
    Draw bounding boxes on a copy of *display_img* and return it as base64 PNG.
    Selected face (index *sel*) gets a 5-rect cyan glow; others get a red outline.
    """
    img = display_img.copy()
    d   = ImageDraw.Draw(img)
    for i, face in enumerate(display_faceobjs):
        r = face["facial_area"]
        x, y, w, h = r["x"], r["y"], r["w"], r["h"]
        if i == sel:
            for off in range(5):
                d.rectangle([x - off, y - off, x + w + off, y + h + off], outline="#00ffcc")
        else:
            d.rectangle([x, y, x + w, y + h], outline="#ff4444", width=2)
    return pil_to_png_b64(img)


def make_thumb(face_arr: np.ndarray, size: int) -> str:
    """Convert a DeepFace float32 face array to a square thumbnail PNG (base64)."""
    pil = arr_to_pil(face_arr)
    if size != -1:
        pil = pil.resize((size, size), Image.LANCZOS)
    pil = ImageOps.expand(pil, border=3, fill="#00ffcc")
    return pil_to_png_b64(pil)


def build_display_faceobjs(faceobjs: list, disp_scale: float) -> list:
    """
    Return a copy of *faceobjs* with facial_area coords scaled to display-image space.
    Used for overlay rendering only — identification uses full-res face arrays.
    """
    return [{
        **face,
        "facial_area": {
            "x": int(face["facial_area"]["x"] * disp_scale),
            "y": int(face["facial_area"]["y"] * disp_scale),
            "w": int(face["facial_area"]["w"] * disp_scale),
            "h": int(face["facial_area"]["h"] * disp_scale),
        }
    } for face in faceobjs]


def face_areas_from_display_faceobjs(display_faceobjs: list) -> list:
    """Extract a plain [{x,y,w,h}] list from display_faceobjs for the client."""
    return [{k: f["facial_area"][k] for k in ("x","y","w","h")} for f in display_faceobjs]


# ─────────────────────────────────────────────────────────────────────────────
# CELL 6 — In-memory face database
# ─────────────────────────────────────────────────────────────────────────────

FACE_DB: list[dict] = []   # [{"label": str, "embedding": list[float]}, …]
DB_LOCK = threading.Lock()


def cosine_similarity(a, b) -> float:
    a, b  = np.array(a), np.array(b)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom > 0 else 0.0


# ─────────────────────────────────────────────────────────────────────────────
# CELL 7 — Background render worker
# ─────────────────────────────────────────────────────────────────────────────

def _render_worker(session_id: str):
    """
    Background daemon thread: render every face-selection frame AND thumbnail
    for the session and store results in render_cache / thumb_cache.

    Design notes
    ────────────
    • Skips face 0 — it was pre-rendered synchronously during /api/upload or
      /api/set_display_px and is already in the cache.
    • Uses render_generation to detect if a /api/set_display_px call has
      invalidated this worker mid-run.  If the generation has changed, the
      worker exits without writing stale data.
    • Sets session["rendering_idx"] to the currently-in-progress index so
      /api/render_status can report it to the client as "rendering_now".
    • Clears rendering_idx to None between faces and on exit.
    """
    session = _SESSIONS.get(session_id)
    if not session:
        print(f"[{_ts()}] _render_worker [{_sid(session_id)}]: session not found, exiting")
        return

    my_gen   = session.get("render_generation", 0)
    n_faces  = len(session["faceobjs"])

    print(f"[{_ts()}] _render_worker [{_sid(session_id)}]: starting, "
          f"{n_faces} faces, generation={my_gen}")

    for i in range(n_faces):
        # ── Superseded check — abort if a newer set_display_px fired ─────────
        if session.get("render_generation", 0) != my_gen:
            print(f"[{_ts()}] _render_worker [{_sid(session_id)}]: "
                  f"generation changed at face {i}, exiting")
            return

        # ── Skip already-cached faces ─────────────────────────────────────────
        with session["render_lock"]:
            if session["render_cache"][i] is not None:
                continue
            session["rendering_idx"] = i

        t0 = time.time()
        print(f"[{_ts()}] _render_worker [{_sid(session_id)}]: rendering face {i} …")

        # Render the annotated frame (CPU-intensive — done outside the lock)
        frame_b64 = render_frame(
            session["display_img"],
            session["display_faceobjs"],
            i,
        )
        # Render the thumbnail (fast)
        thumb_b64 = make_thumb(session["faceobjs"][i]["face"], THUMB_SIZE)

        elapsed = time.time() - t0
        time.sleep(0)
        # ── Final generation check before writing ─────────────────────────────
        with session["render_lock"]:
            if session.get("render_generation", 0) != my_gen:
                print(f"[{_ts()}] _render_worker [{_sid(session_id)}]: "
                      f"generation changed while writing face {i}, exiting")
                session["rendering_idx"] = None
                return
            session["render_cache"][i] = frame_b64
            session["thumb_cache"][i]  = thumb_b64
            session["rendering_idx"]   = None

        print(f"[{_ts()}] _render_worker [{_sid(session_id)}]: "
              f"face {i} done in {elapsed:.2f}s")

    print(f"[{_ts()}] _render_worker [{_sid(session_id)}]: "
          f"all {n_faces} faces complete")
    if STOP_EVENT.is_set():
        print(f"[{_ts()}] Worker stopping early due to shutdown")
        return


def _reset_render_cache(session: dict):
    """
    Clear render_cache, thumb_cache, and delivered_indices, then bump
    render_generation to invalidate any currently-running _render_worker.
    Called by /api/set_display_px before starting a fresh worker.
    """
    with session["render_lock"]:
        n = len(session["faceobjs"])
        session["render_cache"]      = [None] * n
        session["thumb_cache"]       = [None] * n
        session["delivered_indices"] = set()
        session["rendering_idx"]     = None
        session["render_generation"] = session.get("render_generation", 0) + 1
    print(f"[{_ts()}] _reset_render_cache: cache cleared, "
          f"new generation={session['render_generation']}")


# ─────────────────────────────────────────────────────────────────────────────
# CELL 8 — Flask application
# ─────────────────────────────────────────────────────────────────────────────

app = Flask(__name__)
CORS(app, resources={r"/api/*": {"origins": "*"}})


# ── GET /api/health ────────────────────────────────────────────────────────────

@app.route("/api/health", methods=["GET"])
def health():
    """Liveness probe. Returns current FACE_DB entry count."""
    print(f"[{_ts()}] GET /api/health  →  db_size={len(FACE_DB)}")
    return jsonify({"status": "ok", "face_db_size": len(FACE_DB)})


# ── POST /api/upload ───────────────────────────────────────────────────────────

@app.route("/api/upload", methods=["POST"])
def upload():
    """
    Accept a multipart image upload, run face detection, and return face 0's
    annotated frame + thumbnail immediately.

    Session schema created here:
      image, faceobjs, display_img, display_faceobjs, display_max_px,
      render_cache[i]  — base64 PNG or None (face 0 pre-filled)
      thumb_cache[i]   — base64 PNG or None (face 0 pre-filled)
      render_lock, rendering_idx, delivered_indices, render_generation

    Form fields:
      image          — image file (JPEG/PNG/WEBP …)
      detect_max_px  — int string (optional)
      display_max_px — int string (optional)
    """
    print(f"[{_ts()}] POST /api/upload  content_length={request.content_length}")

    if "image" not in request.files:
        print(f"[{_ts()}] POST /api/upload  ERROR: no image field")
        return jsonify({"error": "No image field in request"}), 400

    detect_max_px  = int(request.form.get("detect_max_px",  DETECT_MAX_PX))
    display_max_px = int(request.form.get("display_max_px", DISPLAY_MAX_PX))
    print(f"[{_ts()}] POST /api/upload  detect_max_px={detect_max_px}  "
          f"display_max_px={display_max_px}")

    file = request.files["image"]
    try:
        image = Image.open(file.stream).convert("RGB")
        print(f"[{_ts()}] POST /api/upload  image opened: {image.size[0]}×{image.size[1]}px")
    except Exception as exc:
        print(f"[{_ts()}] POST /api/upload  ERROR opening image: {exc}")
        return jsonify({"error": f"Cannot open image: {exc}"}), 400

    try:
        original_image, faceobjs = get_face_data(image, detect_max_px)
    except Exception as exc:
        print(f"[{_ts()}] POST /api/upload  ERROR in detection: {exc}")
        return jsonify({"error": f"Detection failed: {exc}"}), 500

    if not faceobjs:
        print(f"[{_ts()}] POST /api/upload  no faces detected")
        return jsonify({"error": "No faces detected in this image"}), 200

    display_img, disp_scale = resize_keep_aspect(original_image, display_max_px)
    display_faceobjs        = build_display_faceobjs(faceobjs, disp_scale)
    disp_w, disp_h          = display_img.size
    print(f"[{_ts()}] POST /api/upload  display_img={disp_w}×{disp_h}px  "
          f"disp_scale={disp_scale:.3f}  faces={len(faceobjs)}")

    # Pre-render face 0 synchronously so the UI can show something immediately
    t0 = time.time()
    frame0_b64 = render_frame(display_img, display_faceobjs, 0)
    thumb0_b64 = make_thumb(faceobjs[0]["face"], THUMB_SIZE)
    print(f"[{_ts()}] POST /api/upload  face 0 rendered in {time.time()-t0:.2f}s")

    # Initialise caches: face 0 pre-filled, rest None
    n = len(faceobjs)
    render_cache       = [None] * n;  render_cache[0] = frame0_b64
    thumb_cache        = [None] * n;  thumb_cache[0]  = thumb0_b64

    session_id = str(uuid.uuid4())
    _SESSIONS[session_id] = {
        "image":             original_image,
        "faceobjs":          faceobjs,
        "display_img":       display_img,
        "display_faceobjs":  display_faceobjs,
        "display_max_px":    display_max_px,
        # Render caches — both indexed by face index
        "render_cache":      render_cache,   # frame PNGs
        "thumb_cache":       thumb_cache,    # thumbnail PNGs
        "render_lock":       threading.Lock(),
        "rendering_idx":     None,
        "delivered_indices": set(),          # frames already sent to the client
        "render_generation": 0,
    }

    conf = faceobjs[0].get("confidence")
    if (conf):
        print(f"[{_ts()}] POST /api/upload  session={_sid(session_id)}  "
          f"conf={conf:.4f}")
    else:
        print(f"[{_ts()}] POST /api/upload  session={_sid(session_id)}  "
          f"conf=N/A")
    # print(f"[{_ts()}] POST /api/upload  session={_sid(session_id)}  "
    #       f"conf={conf:.4f if conf else 'N/A'}")

    return jsonify({
        "session_id": session_id,
        "face_count": n,
        "frame_b64":  frame0_b64,
        "thumb_b64":  thumb0_b64,
        "confidence": round(conf, 4) if conf else None,
        "disp_w":     disp_w,
        "disp_h":     disp_h,
        "face_areas": face_areas_from_display_faceobjs(display_faceobjs),
    })


# ── POST /api/render_all ───────────────────────────────────────────────────────

@app.route("/api/render_all", methods=["POST"])
def render_all():
    """
    Start background rendering of ALL face frames + thumbnails for the session.
    Returns immediately — rendering is non-blocking.

    Request JSON:  { "session_id": "<uuid>" }
    Response JSON: { "status": "started", "face_count": N }
    """
    data       = request.get_json(force=True)
    session_id = data.get("session_id", "")
    session    = _SESSIONS.get(session_id)

    print(f"[{_ts()}] POST /api/render_all  session={_sid(session_id)}")

    if not session:
        print(f"[{_ts()}] POST /api/render_all  ERROR: unknown session")
        return jsonify({"error": "Unknown session_id"}), 404

    n = len(session["faceobjs"])
    print(f"[{_ts()}] POST /api/render_all  launching worker for {n} faces")

    t = threading.Thread(target=_render_worker, args=(session_id,), daemon=True)
    t.start()

    return jsonify({"status": "started", "face_count": n})


# ── GET /api/render_status  (metadata only — no image data) ──────────────────
@app.route("/api/render_status", methods=["GET"])
def render_status():
    session_id = request.args.get("session_id", "")
    session    = _SESSIONS.get(session_id)
    if not session:
        return jsonify({"error": "Unknown session_id"}), 404

    render_lock   = session["render_lock"]
    render_cache  = session["render_cache"]
    rendering_idx = session.get("rendering_idx")
    ready_list    = []

    with render_lock:
        for i, frame_b64 in enumerate(render_cache):
            if frame_b64 is not None:
                ready_list.append(i)

    return jsonify({
        "ready":         ready_list,
        "rendering_now": [rendering_idx] if rendering_idx is not None else [],
        "total":         len(render_cache),
        "frames":        {},   # always empty now — client fetches individually
    })


# ── GET /api/get_frame  (one face at a time) ──────────────────────────────────
@app.route("/api/get_frame", methods=["GET"])
def get_frame():
    session_id = request.args.get("session_id", "")
    idx        = int(request.args.get("face_index", 0))
    session    = _SESSIONS.get(session_id)
    if not session:
        return jsonify({"error": "Unknown session_id"}), 404

    with session["render_lock"]:
        frame_b64 = session["render_cache"][idx]
        thumb_b64 = session["thumb_cache"][idx]

    if not frame_b64:
        return jsonify({"error": "Not ready yet"}), 202

    return jsonify({"frame_b64": frame_b64, "thumb_b64": thumb_b64})


# ── POST /api/set_display_px ───────────────────────────────────────────────────

@app.route("/api/set_display_px", methods=["POST"])
def set_display_px():
    """
    Change the display resolution for an existing session and re-render everything.

    Steps:
    1. Rebuild display_img + display_faceobjs at the new resolution.
    2. Invalidate the render cache (_reset_render_cache bumps generation).
    3. Immediately render the currently-selected face at the new resolution.
    4. Seed that frame + thumb into the cache and mark as delivered.
    5. Start a new _render_worker for all remaining faces.

    Request JSON:
      { "session_id", "display_max_px", "current_face_index" }

    Response JSON:
      { "disp_w", "disp_h", "current_frame_b64", "thumb_b64",
        "confidence", "face_areas" }
    """
    data               = request.get_json(force=True)
    session_id         = data.get("session_id", "")
    new_display_max_px = int(data.get("display_max_px", DISPLAY_MAX_PX))
    current_idx        = int(data.get("current_face_index", 0))

    print(f"[{_ts()}] POST /api/set_display_px  session={_sid(session_id)}  "
          f"display_max_px={new_display_max_px}  current_face={current_idx}")

    session = _SESSIONS.get(session_id)
    if not session:
        print(f"[{_ts()}] POST /api/set_display_px  ERROR: unknown session")
        return jsonify({"error": "Unknown session_id"}), 404

    faceobjs    = session["faceobjs"]
    current_idx = max(0, min(current_idx, len(faceobjs) - 1))

    # Rebuild display image at the new resolution
    original_image          = session["image"]
    display_img, disp_scale = resize_keep_aspect(original_image, new_display_max_px)
    display_faceobjs        = build_display_faceobjs(faceobjs, disp_scale)
    disp_w, disp_h          = display_img.size
    print(f"[{_ts()}] POST /api/set_display_px  new display={disp_w}×{disp_h}px  "
          f"disp_scale={disp_scale:.3f}")

    session["display_img"]      = display_img
    session["display_faceobjs"] = display_faceobjs
    session["display_max_px"]   = new_display_max_px

    # Flush stale cache + bump generation so old worker exits
    _reset_render_cache(session)

    # Immediately render the current face so the UI can show something right away
    t0 = time.time()
    frame_b64 = render_frame(display_img, display_faceobjs, current_idx)
    thumb_b64 = make_thumb(faceobjs[current_idx]["face"], THUMB_SIZE)
    print(f"[{_ts()}] POST /api/set_display_px  face {current_idx} rendered "
          f"in {time.time()-t0:.2f}s")

    with session["render_lock"]:
        session["render_cache"][current_idx] = frame_b64
        session["thumb_cache"][current_idx]  = thumb_b64
        # Mark as delivered so the poll loop won't re-send it
        session["delivered_indices"].add(current_idx)

    # Start background rendering for all remaining faces
    t = threading.Thread(target=_render_worker, args=(session_id,), daemon=True)
    t.start()
    print(f"[{_ts()}] POST /api/set_display_px  background worker started")

    conf = faceobjs[current_idx].get("confidence")
    return jsonify({
        "disp_w":            disp_w,
        "disp_h":            disp_h,
        "current_frame_b64": frame_b64,
        "thumb_b64":         thumb_b64,
        "confidence":        round(conf, 4) if conf else None,
        "face_areas":        face_areas_from_display_faceobjs(display_faceobjs),
    })


# ── POST /api/select ───────────────────────────────────────────────────────────

@app.route("/api/select", methods=["POST"])
def select_face():
    """
    Return the frame + thumbnail for the selected face index.

    Serves from render_cache / thumb_cache if available (avoids re-rendering).
    Falls back to a fresh render for uncached faces (shouldn't happen normally
    since the client guards on readySet, but handles edge cases gracefully).

    Request JSON:  { "session_id": "<uuid>", "face_index": <int> }
    Response JSON: { "frame_b64", "thumb_b64", "confidence" }
    """
    data       = request.get_json(force=True)
    session_id = data.get("session_id", "")
    idx        = int(data.get("face_index", 0))
    session    = _SESSIONS.get(session_id)

    print(f"[{_ts()}] POST /api/select  session={_sid(session_id)}  face={idx}")

    if not session:
        print(f"[{_ts()}] POST /api/select  ERROR: unknown session")
        return jsonify({"error": "Unknown session_id"}), 404

    faceobjs         = session["faceobjs"]
    display_img      = session["display_img"]
    display_faceobjs = session["display_faceobjs"]
    idx = max(0, min(idx, len(faceobjs) - 1))

    with session["render_lock"]:
        cached_frame = session["render_cache"][idx]
        cached_thumb = session["thumb_cache"][idx]

    if cached_frame and cached_thumb:
        print(f"[{_ts()}] POST /api/select  serving face {idx} from cache")
        frame_b64 = cached_frame
        thumb_b64 = cached_thumb
    else:
        print(f"[{_ts()}] POST /api/select  cache miss for face {idx}, rendering now")
        t0        = time.time()
        frame_b64 = render_frame(display_img, display_faceobjs, idx)
        thumb_b64 = make_thumb(faceobjs[idx]["face"], THUMB_SIZE)
        print(f"[{_ts()}] POST /api/select  face {idx} rendered in {time.time()-t0:.2f}s")

    conf = faceobjs[idx].get("confidence")
    print("Serving...");
    return jsonify({
        "frame_b64": frame_b64,
        "thumb_b64": thumb_b64,
        "confidence": round(conf, 4) if conf else None,
    })


# ── POST /api/add_to_db ────────────────────────────────────────────────────────

@app.route("/api/add_to_db", methods=["POST"])
def add_to_db():
    """
    Embed the selected face with Facenet512 and store it in FACE_DB.

    Request JSON:  { "session_id", "face_index", "label" }
    Response JSON: { "message", "db_size" }
    """
    data       = request.get_json(force=True)
    session_id = data.get("session_id", "")
    idx        = int(data.get("face_index", 0))
    label      = data.get("label", f"Person_{len(FACE_DB)+1}")

    print(f"[{_ts()}] POST /api/add_to_db  session={_sid(session_id)}  "
          f"face={idx}  label='{label}'")

    session = _SESSIONS.get(session_id)
    if not session:
        print(f"[{_ts()}] POST /api/add_to_db  ERROR: unknown session")
        return jsonify({"error": "Unknown session_id"}), 404

    faceobjs = session["faceobjs"]
    idx      = max(0, min(idx, len(faceobjs) - 1))
    face_arr = faceobjs[idx]["face"]

    tmp_path = f"/tmp/_face_{uuid.uuid4().hex}.jpg"
    arr_to_pil(face_arr).save(tmp_path)

    t0 = time.time()
    try:
        result    = df.represent(tmp_path, model_name="Facenet512", detector_backend="skip")
        embedding = result[0]["embedding"]
        print(f"[{_ts()}] POST /api/add_to_db  embedding done in {time.time()-t0:.2f}s  "
              f"dim={len(embedding)}")
    except Exception as exc:
        print(f"[{_ts()}] POST /api/add_to_db  ERROR embedding: {exc}")
        return jsonify({"error": f"Embedding failed: {exc}"}), 500
    finally:
        os.remove(tmp_path)

    with DB_LOCK:
        FACE_DB.append({"label": label, "embedding": embedding})
        db_size = len(FACE_DB)

    print(f"[{_ts()}] POST /api/add_to_db  '{label}' added — db_size={db_size}")
    return jsonify({"message": f"✅ '{label}' added to database.", "db_size": db_size})


# ── POST /api/identify ─────────────────────────────────────────────────────────

@app.route("/api/identify", methods=["POST"])
def identify():
    """
    Embed the selected face and cosine-search FACE_DB for the best match.
    Reports "Unknown" if best similarity < 0.70.

    Request JSON:  { "session_id", "face_index" }
    Response JSON: { "match", "similarity", "message" }
    """
    data       = request.get_json(force=True)
    session_id = data.get("session_id", "")
    idx        = int(data.get("face_index", 0))

    print(f"[{_ts()}] POST /api/identify  session={_sid(session_id)}  face={idx}")

    session = _SESSIONS.get(session_id)
    if not session:
        print(f"[{_ts()}] POST /api/identify  ERROR: unknown session")
        return jsonify({"error": "Unknown session_id"}), 404

    with DB_LOCK:
        if not FACE_DB:
            print(f"[{_ts()}] POST /api/identify  database is empty")
            return jsonify({"match": "Unknown", "similarity": 0.0,
                            "message": "⚠️ Database is empty — add some faces first."})

    faceobjs = session["faceobjs"]
    idx      = max(0, min(idx, len(faceobjs) - 1))
    face_arr = faceobjs[idx]["face"]

    tmp_path = f"/tmp/_face_{uuid.uuid4().hex}.jpg"
    arr_to_pil(face_arr).save(tmp_path)

    t0 = time.time()
    try:
        result    = df.represent(tmp_path, model_name="Facenet512", detector_backend="skip")
        query_emb = result[0]["embedding"]
        print(f"[{_ts()}] POST /api/identify  embedding done in {time.time()-t0:.2f}s")
    except Exception as exc:
        print(f"[{_ts()}] POST /api/identify  ERROR embedding: {exc}")
        return jsonify({"error": f"Embedding failed: {exc}"}), 500
    finally:
        os.remove(tmp_path)

    with DB_LOCK:
        best_label, best_sim = "Unknown", -1.0
        for entry in FACE_DB:
            sim = cosine_similarity(query_emb, entry["embedding"])
            if sim > best_sim:
                best_sim, best_label = sim, entry["label"]

    THRESHOLD = 0.70
    if best_sim < THRESHOLD:
        best_label = "Unknown"

    print(f"[{_ts()}] POST /api/identify  best_match='{best_label}'  "
          f"similarity={best_sim:.4f}  threshold={THRESHOLD}")

    return jsonify({
        "match":      best_label,
        "similarity": round(best_sim, 4),
        "message":    f"🔍 Best match: {best_label} (similarity {best_sim:.3f})",
    })


# ─────────────────────────────────────────────────────────────────────────────
# CELL 9 — Cloudflare Quick Tunnel
# ─────────────────────────────────────────────────────────────────────────────

CLOUDFLARED_BIN = "/usr/local/bin/cloudflared"
CLOUDFLARED_URL = (
    "https://github.com/cloudflare/cloudflared/releases/latest/"
    "download/cloudflared-linux-amd64"
)

def shutdown_server():
    print(f"[{_ts()}] Shutting down server...")

    STOP_EVENT.set()

    # Kill cloudflared
    global CLOUDFLARE_PROC
    if CLOUDFLARE_PROC and CLOUDFLARE_PROC.poll() is None:
        print(f"[{_ts()}] Killing cloudflared...")
        CLOUDFLARE_PROC.terminate()
        try:
            CLOUDFLARE_PROC.wait(timeout=5)
        except subprocess.TimeoutExpired:
            CLOUDFLARE_PROC.kill()

    # Try to stop Flask (Werkzeug only)
    try:
        func = request.environ.get('werkzeug.server.shutdown')
        if func:
            func()
    except:
        pass

def ensure_cloudflared() -> str:
    if os.path.isfile(CLOUDFLARED_BIN) and os.access(CLOUDFLARED_BIN, os.X_OK):
        print(f"[{_ts()}] cloudflared already installed at {CLOUDFLARED_BIN}")
        return CLOUDFLARED_BIN
    print(f"[{_ts()}] Downloading cloudflared binary…")
    result = subprocess.run(["wget", "-q", "-O", CLOUDFLARED_BIN, CLOUDFLARED_URL],
                            capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"wget failed: {result.stderr}")
    os.chmod(CLOUDFLARED_BIN, 0o755)
    print(f"[{_ts()}] cloudflared installed → {CLOUDFLARED_BIN}")
    return CLOUDFLARED_BIN


def start_cloudflare_tunnel(port: int, timeout: int = 60) -> str | None:
    global CLOUDFLARE_PROC
    """Launch cloudflared and parse the *.trycloudflare.com URL from stderr."""
    bin_path    = ensure_cloudflared()
    proc        = subprocess.Popen(
        [bin_path, "tunnel", "--url", f"http://localhost:{port}"],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True,
    )
    CLOUDFLARE_PROC = proc
    url_pattern = re.compile(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com")
    deadline    = time.time() + timeout
    while time.time() < deadline:
        line = proc.stderr.readline()
        if not line:
            print(f"[{_ts()}] ⚠️  cloudflared process exited unexpectedly.")
            break
        match = url_pattern.search(line)
        if match:
            return match.group(0)
    print(f"[{_ts()}] ⚠️  Timed out waiting for Cloudflare tunnel URL.")
    return None


# ─────────────────────────────────────────────────────────────────────────────
# CELL 10 — Entry point
# ─────────────────────────────────────────────────────────────────────────────
def find_free_port():
  global FLASK_PORT
  while True:
      with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
          # connect_ex returns 0 if the port is open/busy
          if s.connect_ex(('0.0.0.0', FLASK_PORT)) != 0:
              return FLASK_PORT
          FLASK_PORT += 1

def run_server():
    """Start Flask dev server in a daemon thread."""
    app.run(host="0.0.0.0", port=find_free_port(), use_reloader=False, threaded=True)


if __name__ == "__main__" or True:
    try:
        server_thread = threading.Thread(target=run_server, daemon=True)
        server_thread.start()
        print(f"[{_ts()}] Flask server started on http://localhost:{FLASK_PORT}")
        time.sleep(1)

        print(f"[{_ts()}] Opening Cloudflare Quick Tunnel… (may take ~15 s)")
        public_url = start_cloudflare_tunnel(FLASK_PORT)

        if public_url:
            print("\n" + "═" * 60)
            print(f"  ✅  API is live at:\n      {public_url}")
            print(f"\n  Paste this URL into the GitHub Pages UI.")
            print("═" * 60 + "\n")
        else:
            print("❌  Could not obtain tunnel URL.")

        while not STOP_EVENT.is_set():
            time.sleep(1)
    except KeyboardInterrupt:
        print(f"[{_ts()}] KeyboardInterrupt received")
    finally:
        shutdown_server()